# Smart Noise Cancellation Fuzzy Controller
This notebook explains each implementation step.

### Step 1: Install Required Libraries

In [ ]:
!pip install scikit-fuzzy matplotlib numpy

###Step 2: Import Libraries

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt

###Step 3: Define Input and Output Variables

In [ ]:
# Input 1: Ambient Noise Level (dB)
noise = ctrl.Antecedent(np.arange(0, 101, 1), 'noise')

# Input 2: Noise Frequency (Hz)
frequency = ctrl.Antecedent(np.arange(20, 2001, 1), 'frequency')

# Input 3: Signal-to-Noise Ratio (dB)
snr = ctrl.Antecedent(np.arange(-10, 31, 1), 'snr')

# Output: Noise Cancellation Gain (%)
cancellation = ctrl.Consequent(np.arange(0, 101, 1), 'cancellation')

###Step 4: Define Membership Functions

####Ambient Noise Level

In [ ]:
noise['low'] = fuzz.trimf(noise.universe, [0, 0, 40])
noise['medium'] = fuzz.trimf(noise.universe, [20, 50, 80])
noise['high'] = fuzz.trimf(noise.universe, [60, 100, 100])

In [ ]:
noise.view()

####Noise Frequency

In [ ]:
frequency['low'] = fuzz.trimf(frequency.universe, [20, 20, 700])
frequency['medium'] = fuzz.trimf(frequency.universe, [400, 1000, 1600])
frequency['high'] = fuzz.trimf(frequency.universe, [1300, 2000, 2000])

In [ ]:
frequency.view()

####Signal-to-Noise Ratio

In [ ]:
snr['low'] = fuzz.trimf(snr.universe, [-10, -10, 5])
snr['medium'] = fuzz.trimf(snr.universe, [0, 10, 20])
snr['high'] = fuzz.trimf(snr.universe, [15, 30, 30])

In [ ]:
snr.view()

####Noise Cancellation Gain

In [ ]:
cancellation['very_low'] = fuzz.trimf(cancellation.universe, [0, 0, 15])

cancellation['low'] = fuzz.trimf(cancellation.universe, [10, 25, 40])

cancellation['medium'] = fuzz.trimf(cancellation.universe, [35, 50, 65])

cancellation['high'] = fuzz.trimf(cancellation.universe, [60, 75, 90])

cancellation['very_high'] = fuzz.trimf(cancellation.universe, [85, 100, 100])

In [ ]:
cancellation.view()

###Step 6: Create Rule Base

In [ ]:
rule1 = ctrl.Rule(
    noise['high'] &
    frequency['high'] &
    snr['low'],
    cancellation['very_high']
)

rule2 = ctrl.Rule(
    noise['high'] &
    frequency['medium'],
    cancellation['high']
)

rule3 = ctrl.Rule(
    noise['medium'] &
    frequency['high'],
    cancellation['medium']
)

rule4 = ctrl.Rule(
    noise['low'],
    cancellation['very_low']
)

rule5 = ctrl.Rule(
    snr['high'],
    cancellation['very_low']
)

rule6 = ctrl.Rule(
    noise['high'] &
    snr['high'],
    cancellation['low']
)

rule7 = ctrl.Rule(
    noise['medium'] &
    snr['medium'],
    cancellation['low']
)

rule8 = ctrl.Rule(
    noise['medium'] &
    snr['low'],
    cancellation['medium']
)

###Step 7: Create Controller

In [ ]:
cancellation_ctrl = ctrl.ControlSystem([
    rule1,
    rule2,
    rule3,
    rule4,
    rule5,
    rule6,
    rule7,
    rule8
])

system = ctrl.ControlSystemSimulation(cancellation_ctrl)

###Step 8: Provide Sensor Values

In [ ]:
system.input['noise'] = 70

system.input['frequency'] = 900

system.input['snr'] = 5

###Step 9: Run Fuzzy Inference
The default defuzzification method for a Consequent in scikit-fuzzy is centroid.

In [ ]:
system.compute()

In [ ]:
print(system.output['cancellation'])

###Step 10: Display Result

In [ ]:
print("Recommended Noise Cancellation Gain:")

print(round(system.output['cancellation'],2),"%")

###Step 11: Show Defuzzification Graph

In [ ]:
cancellation.view(sim=system)

plt.show()

###Step-12 Try Different Test Cases
####Test Case 1

In [ ]:
system.input['noise'] = 40
system.input['frequency'] = 300
system.input['snr'] = 15
system.compute()

print(system.output['cancellation'])

###Step-13: Apply different Defuzzification methods and observe the result and compare.
| Method       | Meaning                                          | Typical use                                 |
| ------------ | ------------------------------------------------ | ------------------------------------------- |
| **centroid** | Center of Area / Center of Gravity               | ⭐ Most commonly used                        |
| **bisector** | Divides the aggregated area into two equal areas | Alternative to centroid                     |
| **mom**      | Mean of Maximum                                  | When maximum membership region is important |
| **som**      | Smallest of Maximum                              | Selects the smallest x among maxima         |
| **lom**      | Largest of Maximum                               | Selects the largest x among maxima          |

####1. Centroid
It finds the balance point of the aggregated fuzzy output.

In [ ]:
cancellation.defuzzify_method = 'centroid'
system = ctrl.ControlSystemSimulation(cancellation_ctrl)
system.input['noise'] = 70
system.input['frequency'] = 900
system.input['snr'] = 5
system.compute()
print(round(system.output['cancellation'],2),"%")

In [ ]:
cancellation.view(sim=system)

plt.show()

####2. Bisector

Finds the point that divides the total area under the aggregated membership function into two equal areas.

Think of it as:"Where can I cut the fuzzy output so that the area on both sides is equal?"

In [ ]:
cancellation.defuzzify_method = 'bisector'
system = ctrl.ControlSystemSimulation(cancellation_ctrl)
system.input['noise'] = 70
system.input['frequency'] = 900
system.input['snr'] = 5
system.compute()
print(round(system.output['cancellation'],2),"%")

In [ ]:
cancellation.view(sim=system)

plt.show()

####3. MOM — Mean of Maximum

Finds all points having the maximum membership value and takes their average.

For example:Maximum membership occurs at:

60, 61, 62, 63

MOM = (60+61+62+63)/4
    = 61.5

In [ ]:
cancellation.defuzzify_method = 'mom'
system = ctrl.ControlSystemSimulation(cancellation_ctrl)
system.input['noise'] = 70
system.input['frequency'] = 900
system.input['snr'] = 5
system.compute()
print(round(system.output['cancellation'],2),"%")

In [ ]:
cancellation.view(sim=system)

plt.show()

####4. SOM — Smallest of Maximum

Finds the points having maximum membership and chooses the smallest one.
Maximum points:

60, 61, 62, 63

SOM = 60

In [ ]:
cancellation.defuzzify_method = 'som'
system = ctrl.ControlSystemSimulation(cancellation_ctrl)
system.input['noise'] = 70
system.input['frequency'] = 900
system.input['snr'] = 5
system.compute()
print(round(system.output['cancellation'],2),"%")

In [ ]:
cancellation.view(sim=system)

plt.show()

####5. LOM — Largest of Maximum

Same idea, but chooses the largest maximum point.
Maximum points:

60, 61, 62, 63

LOM = 63

In [ ]:
cancellation.defuzzify_method = 'lom'
system = ctrl.ControlSystemSimulation(cancellation_ctrl)
system.input['noise'] = 70
system.input['frequency'] = 900
system.input['snr'] = 5
system.compute()
print(round(system.output['cancellation'],2),"%")

In [ ]:
cancellation.view(sim=system)

plt.show()

Using For Loop

In [ ]:
methods = ['centroid', 'bisector', 'mom', 'som', 'lom']

for method in methods:
    cancellation.defuzzify_method = method

    system = ctrl.ControlSystemSimulation(cancellation_ctrl)

    system.input['noise'] = 70
    system.input['frequency'] = 900
    system.input['snr'] = 5

    system.compute()

    print(method, system.output['cancellation'])

**Conclusion**: There exist different outputs for different defuzz methods. To identify the best amongst them we need real world data.

# Milestone 3

This section contains Phase 4 - Testing, Phase 5 - Analysis and Phase 6 - User Interface for the Smart Noise Cancellation Fuzzy Controller.


# Phase 4 - Testing

The fuzzy controller is tested using 10 different combinations of ambient noise level, noise frequency and signal-to-noise ratio (SNR).

The purpose of testing is to observe how the recommended noise cancellation gain changes for different environmental conditions.


In [ ]:
# Use centroid method for testing
cancellation.defuzzify_method = 'centroid'

test_cases = [
    (20, 300, 15),
    (30, 500, 5),
    (40, 300, 15),
    (50, 1000, 10),
    (50, 1500, 5),
    (60, 1500, 5),
    (70, 900, 5),
    (70, 1500, 20),
    (80, 1500, 5),
    (90, 1000, 10)
]

print("Test Case | Noise(dB) | Frequency(Hz) | SNR(dB) | Cancellation Gain(%)")
print("-" * 78)

results = []

for i, (n, f, s) in enumerate(test_cases, 1):
    test_system = ctrl.ControlSystemSimulation(cancellation_ctrl)

    test_system.input['noise'] = n
    test_system.input['frequency'] = f
    test_system.input['snr'] = s

    test_system.compute()

    result = test_system.output['cancellation']
    results.append((i, n, f, s, result))

    print(f"{i:9} | {n:9} | {f:13} | {s:7} | {result:20.2f}")


## Testing Observation

The test cases represent low, medium and high noise conditions with different frequencies and SNR values.

In general, lower noise conditions require lower cancellation gain, while stronger noise conditions can require higher cancellation. The exact output also depends on frequency and SNR because the fuzzy rules consider these inputs together.

The numerical output obtained from the controller should be used as the final observation for each test case.


# Phase 5 - Analysis

## Controller Behaviour

The fuzzy controller changes the noise cancellation gain according to ambient noise level, noise frequency and signal-to-noise ratio.

When the noise level is low, the controller recommends very low cancellation gain because strong cancellation is not necessary.

When the noise level becomes high, the recommended cancellation gain generally increases. High-frequency noise combined with low SNR can result in stronger cancellation.

A high SNR indicates that the useful signal is relatively strong compared with the noise. Therefore, excessive cancellation is avoided in such situations.

The fuzzy controller gives a gradual response instead of only ON/OFF decisions. This makes it suitable for varying noise conditions.

## Strengths

1. The controller can handle multiple input conditions.
2. Fuzzy logic provides a gradual response instead of sudden switching.
3. The system does not require a precise mathematical model of the noise environment.
4. Different defuzzification methods can be compared.
5. Membership functions and fuzzy rules can be modified easily.

## Weaknesses

1. The controller currently uses predefined membership functions.
2. The rule base contains only 8 rules.
3. Some combinations of input conditions may not be fully represented by the existing rule base.
4. The current system uses simulated input values instead of real-time microphone data.
5. The selected membership ranges may not represent every real-world noise environment.

## Scope of Improvement

1. Add more fuzzy rules to cover more input combinations.
2. Use real-time microphone data for noise measurement.
3. Tune membership functions using experimental or real-world data.
4. Include additional useful noise characteristics if required.
5. Connect the controller with an actual noise cancellation system.
6. Improve the user interface for real-time monitoring and control.


# Phase 6 - User Interface

An interactive user interface is developed using `ipywidgets`.

The user can select the ambient noise level, noise frequency and SNR using sliders. After pressing the **Calculate** button, the fuzzy controller displays the recommended noise cancellation gain.


In [ ]:
# Run this cell if ipywidgets is not installed
!pip install ipywidgets


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

noise_input = widgets.FloatSlider(
    value=70,
    min=0,
    max=100,
    step=1,
    description='Noise (dB):',
    style={'description_width': 'initial'},
    continuous_update=False
)

frequency_input = widgets.FloatSlider(
    value=900,
    min=20,
    max=2000,
    step=10,
    description='Frequency (Hz):',
    style={'description_width': 'initial'},
    continuous_update=False
)

snr_input = widgets.FloatSlider(
    value=5,
    min=-10,
    max=30,
    step=1,
    description='SNR (dB):',
    style={'description_width': 'initial'},
    continuous_update=False
)

calculate_button = widgets.Button(
    description='Calculate',
    button_style='primary',
    tooltip='Calculate recommended cancellation gain'
)

output_box = widgets.Output()

def calculate_result(b):
    with output_box:
        clear_output()

        ui_system = ctrl.ControlSystemSimulation(cancellation_ctrl)

        ui_system.input['noise'] = noise_input.value
        ui_system.input['frequency'] = frequency_input.value
        ui_system.input['snr'] = snr_input.value

        try:
            ui_system.compute()
            result = ui_system.output['cancellation']

            print("SMART NOISE CANCELLATION CONTROLLER")
            print("-----------------------------------")
            print(f"Noise Level : {noise_input.value:.1f} dB")
            print(f"Frequency   : {frequency_input.value:.1f} Hz")
            print(f"SNR         : {snr_input.value:.1f} dB")
            print()
            print(f"Recommended Noise Cancellation Gain: {result:.2f}%")

        except Exception:
            print("No valid fuzzy output was produced for this combination.")
            print("Try another input combination.")

calculate_button.on_click(calculate_result)

title = widgets.HTML(
    value="<h2>Smart Noise Cancellation Fuzzy Controller</h2>"
)

instruction = widgets.HTML(
    value="<b>Select the input values and click Calculate.</b>"
)

ui = widgets.VBox([
    title,
    instruction,
    noise_input,
    frequency_input,
    snr_input,
    calculate_button,
    output_box
])

display(ui)


# Milestone 3 - Final Conclusion

The Smart Noise Cancellation Fuzzy Controller was tested using multiple input combinations. The behaviour of the controller was analysed by studying its response to different noise levels, frequencies and SNR values.

The controller provides a flexible output using fuzzy rules instead of fixed ON/OFF decisions. An interactive user interface was also created so that users can enter input conditions and obtain the recommended noise cancellation gain easily.

For future improvement, the controller can be tested with real microphone data and a larger rule base.
